In [ ]:
import sys
import warnings
from collections import Counter

import cirq
import matplotlib.pyplot as plt
from cirq.contrib.svg import SVGCircuit

sys.path.append("../")
import resource_estimation as res
import scripts.layout_figures as lfs

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
_COLORS = {
    "None": "#ffffff",
    "T Factory": "#4ddef1",
    "S Factory": "#fcc084",
    "Data Qubit": "#a4f0c2",
    "Ancilla Patch": "#f5bad6",
    "Distillation": "#E6E6FA",
    "CNOT": "#ed6340",
    "CCZ Factory": "#000000",
}
_PRETTY = True  # I don't think the SVG circuits are pretty, but this is an option


def cast_circuit(circuit):
    """Use Cirq's optional SVG display for a circuit."""
    return SVGCircuit(circuit) if _PRETTY else circuit

# Overview
This notebook shows how to use `resource-superstaq` to estimate quantum resources when using distillation.  For the case of distilling T states only, one can pass in an arbitrary cirq circuit.  To estimate resources using T and/or CCZ states, the circuit must be expressed in terms of T, S, CNOT, H, and CCZ gates.  That is, the circuit has already been cast in terms of Clifford+T+CCZ.  Currently, only movement architectures and the `MovementDistillery` layout support distillation.  The key metrics will be physical qubits, circuit runtime, and physical operation count. The basic flow is shown below:
- Have a `cirq` circuit
- Choose a Movement Architecture
    - Single Species with Movement (`DefaultMovement`) (SSM)
    - Dual Species with Movement (DSM)
- Use the Distillation Layout
    - Use `MovementDistillery` for SSM or DSM
- Compile Clifford + T + CCZ circuit to primitives
    -  Primitives are Gate-like objects with simple physical decompositions
    -  This step includes operations for movement for movement-based architectures
- Estimate Resources
    - Count gates by unrolling gate costs for primitives
    - Extract time by unrolling gate times for primitives

Here we will first give an example using only T-state distillation, then a second example using T and CCZ-state distillation.


# Distillation with T states

## Have a Circuit
For this example, we use a random unitary

In [ ]:
qubits = 3
U = cirq.testing.random_unitary(dim=2**qubits, random_state=7)
circuit = cirq.Circuit(cirq.MatrixGate(U).on(*cirq.LineQubit.range(qubits)))
circuit

## Compile to Clifford + Rz
Each application circuit generally does not come packaged neatly in a fault tolerant gateset, so we use an intermediate compilation step that converts the circuit to the Clifford + Rz gateset using local Cirq-based optimization (via `resource_estimation.compile_gateset.compile_gateset`, which wraps `cirq.optimize_for_target_gateset` for Cirq gatesets). This process can have the effect of magnifying inefficiencies that might have already existed in the input circuit. This is just a demonstration.

In [ ]:
cliff_rz_circuit = res.compile_gateset.compile_gateset(
    circuit,
    gateset=res.compile_gateset.clifford_rz_gateset(),
)

In [ ]:
print(f"Qubits: {cirq.num_qubits(cliff_rz_circuit)}")
print(f"Moments: {len(cliff_rz_circuit)}")

In [ ]:
print(f"Moments:{' ' * (10 - len(str(len(cliff_rz_circuit))))}{len(cliff_rz_circuit)}")
grouped_ops = Counter(
    str(op.gate) if op.gate not in cirq.GateFamily(cirq.Rz) else "Rz"
    for op in cliff_rz_circuit.all_operations()
)
for key, val in grouped_ops.items():
    if "Measurement" in key:
        key = "Measurement"
    print(f"{key}{' ' * (16 - len(key))}{val}")

## Synthesize Circuit
We can use `pygridsynth` to compile the Rz gates generated in the previous step to strings of H, S, and T. The input parameter epsilon plays the main role in determining the length of these strings. In practice, we can choose this parameter based on the number Rz gates in the circuit and the program fidelity we would like. To keep things simple, we choose a program fidelity of 99%. The higher the requested fidelity, the more expensive the synthesis becomes.

In [ ]:
eps = 1 - 0.99 ** (1 / grouped_ops["Rz"])
synthesized_circuit = res.compile_gateset.compile_gateset(
    cliff_rz_circuit,
    gateset=res.compile_gateset.clifford_t_gateset(atol=eps),
)

In [ ]:
print(f"Moments:{' ' * 12}{len(synthesized_circuit)}")
grouped_ops = Counter(str(op.gate) for op in synthesized_circuit.all_operations())
for key, val in grouped_ops.items():
    if "Measurement" in key:
        key = "Measurement"
    print(f"{key}{' ' * (20 - len(key))}{val}")

## Choose an Architecture

In [ ]:
ssm = res.ftqc.DefaultMovement(
    d=11,  # Rotated Surface Code code distance
    idling=False,  # Include Syndrome Extraction on idling qubits in compiled circuit
    post_op_correction=True,  # Turn on or off Syndrome Extraction after transversal operations
    syndrome_rounds=1,  # Rounds of Syndrome Extraction after transversal operations
    cultivation_repetition=1,  # Expected repetitions of the cultivation circuit to get a successful T state
    distillation_repetition=1,  # Expected repetitions of the distillation circuit to get a successful T state
)

## Choose a Layout

In [ ]:
# Use a layout for distillation
distillation_layout = res.ftqc.layout.MovementDistillery(
    input_circuit=synthesized_circuit, num_t_factories=3
)
fig, ax, im = lfs.plot_layout(distillation_layout, category_colors=_COLORS, transpose=True)
# Use a layout for cultivation
cultivation_layout = res.ftqc.MovementLayout(input_circuit=synthesized_circuit, num_t_factories=3)
fig, ax, im = lfs.plot_layout(cultivation_layout, category_colors=_COLORS, transpose=True)

## FT Compile

In [ ]:
# Compile the circuit for distillation
primitive_circuit_for_distillation = res.ftqc.ft_compile(
    layout=distillation_layout, arc=ssm, verbose=True
)

In [ ]:
# Compile the circuit for cultivation
primitive_circuit_for_cultivation = res.ftqc.ft_compile(
    layout=cultivation_layout, arc=ssm, verbose=True
)

## Estimate Resources

In [ ]:
# Distillation estimates
estimator = res.ftqc.ResourceEstimator(arc=ssm)
gate_cost = estimator.parallel_circuit_cost(primitive_circuit_for_distillation, pretty=True)
circuit_time = estimator.parallel_circuit_time(primitive_circuit_for_distillation)
physical_qubits = estimator.physical_qubits(primitive_circuit_for_distillation)

In [ ]:
print("Gates")
for op, count in sorted(gate_cost.items(), key=lambda x: x[1], reverse=True):
    print(f"{op}{' ' * (25 - len(op))}{count:.2e}")
print("*" * 33)
print("Metrics")
print(f"Time (μs){' ' * 16}{circuit_time:.2e}\nQubits{' ' * 19}{physical_qubits:.2e}")
print("*" * 33)

In [ ]:
# Cultivation estimates
gate_cost = estimator.parallel_circuit_cost(primitive_circuit_for_cultivation, pretty=True)
circuit_time = estimator.parallel_circuit_time(primitive_circuit_for_cultivation)
physical_qubits = estimator.physical_qubits(primitive_circuit_for_cultivation)

In [ ]:
print("Gates")
for op, count in sorted(gate_cost.items(), key=lambda x: x[1], reverse=True):
    print(f"{op}{' ' * (25 - len(op))}{count:.2e}")
print("*" * 33)
print("Metrics")
print(f"Time (μs){' ' * 16}{circuit_time:.2e}\nQubits{' ' * 19}{physical_qubits:.2e}")
print("*" * 33)

# Distillation with T and CCZ states

## Have a Circuit
For this example, we use a random unitary made of only Clifford+T+CCZ gates.

In [ ]:
qubits = 5


def make_test_circuit(num_qubits) -> cirq.Circuit:
    circuit = cirq.testing.random_circuit(
        cirq.LineQubit.range(num_qubits),
        10,
        0.6,
        {cirq.T: 1, cirq.S: 1, cirq.CNOT: 2, cirq.H: 1, cirq.CCZ: 3},
        44,
    )
    return circuit


circuit = make_test_circuit(qubits)
cast_circuit(circuit)  # This circuit is only composed of Cliffords, Ts, and CCZs

## Choose a Movement Architecture

In [ ]:
ssm = res.ftqc.DefaultMovement(
    d=11,  # Rotated Surface Code code distance
    idling=False,  # Include Syndrome Extraction on idling qubits in compiled circuit
    post_op_correction=True,  # Turn on or off Syndrome Extraction after transversal operations
    syndrome_rounds=1,  # Rounds of Syndrome Extraction after transversal operations
    cultivation_repetition=5,  # Expected repetitions of the cultivation circuit to get a successful T state
    distillation_repetition=1,  # Expected repetitions needed for a successful T or CCZ state
)
estimator = res.ftqc.ResourceEstimator(arc=ssm)  # Create an estimator object
# Notice the T-states used as input for distillation come from cultivation

## Choose a Layout

In [ ]:
layout = res.ftqc.layout.MovementDistillery(
    input_circuit=circuit, num_t_factories=2, num_ccz_factories=1
)
fig, ax, im = lfs.plot_layout(layout, category_colors=_COLORS, transpose=True)

## FT Compile

In [ ]:
# Recompile the circuit in terms of primitives
primitive_circuit = res.ftqc.ft_compile(layout=layout, arc=ssm, verbose=True)
cast_circuit(primitive_circuit)

In [ ]:
# Primitive gate counts
primitive_gate_counts = Counter(
    [
        "Measure" if op in cirq.GateFamily(cirq.MeasurementGate) else str(op.gate)
        for op in primitive_circuit.all_operations()
    ]
)
print("Primitive gate counts")
for op, count in sorted(primitive_gate_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{op}{' ' * (25 - len(op))}{count}")

In [ ]:
# Primitives in the critical path
primitive_cost = estimator.critical_path(primitive_circuit)
primitive_counts = Counter(
    [
        "Measure" if primitive in cirq.GateFamily(cirq.MeasurementGate) else str(primitive.gate)
        for primitive in primitive_cost
    ]
)
print("Critical path primitives")
for op, count in sorted(primitive_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{op}{' ' * (25 - len(op))}{count}")

In [ ]:
# We see distil_t is the largest cost in time
op_times = [
    ssm._distil_cost("CCZ")["op_time"],
    ssm._distil_cost("T")["op_time"],
    ssm._s_cost["op_time"],
    ssm._cnot_cost["op_time"],
    ssm._h_cost["op_time"],
    ssm._measure_cost["op_time"],
]
op_names = [
    "distil_ccz",
    "distil_t",
    "s",
    "cnot",
    "h",
    "measure",
]
f, ax = plt.subplots()
ax.bar(op_names, op_times)
ax.set_xlabel("Op")
ax.set_ylabel(r"Time ($\mu$s)")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## Estimate Resources

In [ ]:
# Extract the various costs
gate_cost = estimator.parallel_circuit_cost(primitive_circuit, pretty=True)
circuit_time = estimator.parallel_circuit_time(primitive_circuit)
physical_qubits = estimator.physical_qubits(primitive_circuit)

In [ ]:
print("Gates")
for op, count in sorted(gate_cost.items(), key=lambda x: x[1], reverse=True):
    print(f"{op}{' ' * (25 - len(op))}{count}")
print("*" * 33)
print("Metrics")
print(f"Time (μs){' ' * 16}{circuit_time:.2e}\nQubits{' ' * 19}{physical_qubits:.2e}")
print("*" * 33)